# LAB 04 — Decision Tree: từ dự đoán đến quy tắc hỗ trợ

**DNU Learning Analytics Lab**

- **Họ tên:** ...
- **MSSV:** ...
- **Lớp:** ...

> **Câu hỏi trung tâm:** Decision Tree đưa ra dự đoán bằng những câu hỏi nào, và làm thế nào để biến một cây quyết định thành các quy tắc có thể giải thích?

Lab 04 dùng **cùng bài toán, cùng feature, cùng target và cùng train/test split của Lab 02–03**.

- K-NN: “Bạn giống ai?”
- Naive Bayes: “Khả năng thuộc lớp nào?”
- Decision Tree: **“Những quyết định nào dẫn đến kết quả?”**

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path("data/StudentPerformanceFactors.csv")
print("Environment ready! Dataset exists:", DATA_PATH.exists())

# Mission 1 — Back to DNU case study

Ta giữ nguyên bài toán của Lab 02–03:

```text
Needs_Support = 1 nếu Exam_Score < 65
```

Feature chính:

```text
Hours_Studied
Attendance
Previous_Scores
Sleep_Hours
```

**Quan trọng:** Không dùng `Exam_Score` làm feature vì target được tạo trực tiếp từ biến này.

**[TRẢ LỜI]**
1. Vì sao dùng `Exam_Score` làm feature sẽ gây target leakage?
2. Vì sao Lab 04 phải giữ cùng train/test split với Lab 02–03?

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError("Run: python scripts/download_data.py")

df = pd.read_csv(DATA_PATH).copy()
df["Needs_Support"] = (df["Exam_Score"] < 65).astype(int)

feature_cols = [
    "Hours_Studied",
    "Attendance",
    "Previous_Scores",
    "Sleep_Hours",
]

assert "Exam_Score" not in feature_cols

model_df = df[feature_cols + ["Needs_Support"]].dropna().copy()
X = model_df[feature_cols]
y = model_df["Needs_Support"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
display(y_train.value_counts().rename("count").to_frame())
display(y_train.value_counts(normalize=True).rename("ratio").to_frame())

# Mission 2 — Baseline Decision Tree

Huấn luyện cây đầu tiên với thiết lập mặc định.

> Ở bước này **không scale dữ liệu**. Decision Tree chia dữ liệu theo ngưỡng của từng feature, nên scaling không phải bước bắt buộc như K-NN.

**[TRẢ LỜI]**
- Baseline tree có độ sâu bao nhiêu?
- Có bao nhiêu node và leaf?
- Theo bạn, một cây rất sâu có dễ giải thích không?

In [ ]:
tree_baseline = DecisionTreeClassifier(random_state=42)
tree_baseline.fit(X_train, y_train)

print("Tree depth:", tree_baseline.get_depth())
print("Number of leaves:", tree_baseline.get_n_leaves())
print("Number of nodes:", tree_baseline.tree_.node_count)

# Mission 3 — Evaluate beyond Accuracy

Đánh giá bằng:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix

Đặc biệt chú ý:

> **False Negative (FN): sinh viên thực tế cần hỗ trợ nhưng mô hình dự đoán không cần hỗ trợ.**

**[TRẢ LỜI]**
1. Baseline tree có bao nhiêu False Negative?
2. Nếu mục tiêu là hạn chế bỏ sót sinh viên cần hỗ trợ, metric nào cần được chú ý?
3. Vì sao chỉ nhìn Accuracy là chưa đủ?

In [ ]:
def evaluate_binary(name, model, X_eval, y_eval, show_cm=True):
    pred = model.predict(X_eval)
    cm = confusion_matrix(y_eval, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    result = {
        "model": name,
        "accuracy": accuracy_score(y_eval, pred),
        "precision": precision_score(y_eval, pred, zero_division=0),
        "recall": recall_score(y_eval, pred, zero_division=0),
        "f1": f1_score(y_eval, pred, zero_division=0),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }

    if show_cm:
        ConfusionMatrixDisplay.from_predictions(
            y_eval, pred,
            display_labels=["No Support", "Needs Support"]
        )
        plt.title(name)
        plt.show()

    return result, pred

baseline_result, baseline_pred = evaluate_binary(
    "Decision Tree baseline",
    tree_baseline,
    X_test,
    y_test
)

display(pd.DataFrame([baseline_result]))
print(classification_report(
    y_test,
    baseline_pred,
    target_names=["No Support", "Needs Support"],
    zero_division=0
))

# Mission 4 — Read the Tree: từ cây sang quy tắc

Trong bài giảng, ta đã dùng ví dụ **Love AI** để tự xây cây bằng tay.

Bây giờ ta dùng cùng tư duy đó trên Student Performance Factors.

Để cây dễ đọc, tạo một **explainable tree** với `max_depth=3`.

### Nhiệm vụ

1. Xác định **Root Node**.
2. Viết lại ít nhất **2 đường Root → Leaf** thành quy tắc bằng ngôn ngữ tự nhiên.
3. Nhận xét: cây ngắn hơn có đánh đổi điều gì so với baseline tree?

**[TRẢ LỜI — RULE 1]** ...

**[TRẢ LỜI — RULE 2]** ...

In [ ]:
tree_explain = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
tree_explain.fit(X_train, y_train)

plt.figure(figsize=(20, 10))
plot_tree(
    tree_explain,
    feature_names=feature_cols,
    class_names=["No Support", "Needs Support"],
    filled=True,
    rounded=True,
    proportion=True,
    fontsize=9
)
plt.title("Explainable Decision Tree — max_depth=3")
plt.show()

print("Text rules:")
print(export_text(tree_explain, feature_names=feature_cols))

root_feature_index = tree_explain.tree_.feature[0]
root_feature = feature_cols[root_feature_index]
root_threshold = tree_explain.tree_.threshold[0]
print(f"Root node: {root_feature} <= {root_threshold:.3f}")

# Mission 5 — Complexity matters: cây sâu hơn có luôn tốt hơn?

Thử:

```python
max_depth = [2, 3, 5, None]
```

Ghi lại:

- Train Accuracy
- Test Accuracy
- Recall
- F1
- Số node
- Số leaf

**[TRẢ LỜI]**
1. Khi cây sâu hơn, Train Accuracy thay đổi thế nào?
2. Test Accuracy/F1 có tăng mãi không?
3. Dấu hiệu nào gợi ý overfitting?
4. Nếu phải trình bày mô hình cho cố vấn học tập, bạn sẽ ưu tiên cây nào và vì sao?

In [ ]:
depth_rows = []

for depth in [2, 3, 5, None]:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    depth_rows.append({
        "max_depth": str(depth),
        "train_accuracy": accuracy_score(y_train, train_pred),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_recall": recall_score(y_test, test_pred, zero_division=0),
        "test_f1": f1_score(y_test, test_pred, zero_division=0),
        "nodes": model.tree_.node_count,
        "leaves": model.get_n_leaves(),
        "actual_depth": model.get_depth(),
    })

depth_results = pd.DataFrame(depth_rows)
display(depth_results)

plt.figure(figsize=(8, 5))
plt.plot(depth_results["max_depth"], depth_results["train_accuracy"], marker="o", label="Train Accuracy")
plt.plot(depth_results["max_depth"], depth_results["test_accuracy"], marker="o", label="Test Accuracy")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Mission 6 — Gini vs Entropy

Trong ví dụ Love AI, ta đã tính **Gini** và **Entropy / Information Gain** bằng tay.

Bây giờ hãy xem hai criterion này tạo ra cây như thế nào trên dữ liệu thật.

**[TRẢ LỜI]**
1. Accuracy, Recall và F1 có khác nhau nhiều không?
2. Root Node có giống nhau không?
3. Ba split đầu tiên có giống nhau không?
4. Hai cây có thể có hiệu năng gần giống nhau nhưng cấu trúc khác nhau không?

In [ ]:
criterion_rows = []
criterion_models = {}

for criterion in ["gini", "entropy"]:
    model = DecisionTreeClassifier(
        criterion=criterion,
        max_depth=4,
        random_state=42
    )
    model.fit(X_train, y_train)
    criterion_models[criterion] = model

    result, pred = evaluate_binary(
        f"Decision Tree ({criterion})",
        model,
        X_test,
        y_test,
        show_cm=False
    )

    root_idx = model.tree_.feature[0]
    result["root_feature"] = feature_cols[root_idx]
    result["root_threshold"] = model.tree_.threshold[0]
    result["depth"] = model.get_depth()
    result["nodes"] = model.tree_.node_count
    criterion_rows.append(result)

criterion_results = pd.DataFrame(criterion_rows)
display(criterion_results[
    ["model", "accuracy", "precision", "recall", "f1",
     "root_feature", "root_threshold", "depth", "nodes"]
])

for criterion, model in criterion_models.items():
    print("\n====", criterion.upper(), "====")
    print(export_text(model, feature_names=feature_cols, max_depth=2))

# Mission 7 — Small GridSearch: tuning nhưng không biến thành black box

Dùng một không gian tham số nhỏ:

- `criterion`
- `max_depth`
- `min_samples_split`
- `min_samples_leaf`

Mục tiêu không phải “quét càng nhiều càng tốt”, mà quan sát cách hyperparameter kiểm soát độ phức tạp.

Ở đây dùng `scoring="f1"` để không chỉ tối ưu Accuracy.

**[TRẢ LỜI]**
1. `best_params_` là gì?
2. Best tree sâu bao nhiêu?
3. So với baseline, số node giảm hay tăng?
4. Best CV score có giống Test F1 không? Vì sao không nên xem chúng là cùng một con số?

In [ ]:
param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
}

grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_tree = grid.best_estimator_

print("Best params:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 4))
print("Best tree depth:", best_tree.get_depth())
print("Best tree nodes:", best_tree.tree_.node_count)
print("Best tree leaves:", best_tree.get_n_leaves())

best_result, best_pred = evaluate_binary(
    "Decision Tree — GridSearch best",
    best_tree,
    X_test,
    y_test
)
display(pd.DataFrame([baseline_result, best_result])[
    ["model", "accuracy", "precision", "recall", "f1", "fn", "fp"]
])

# Mission 8 — Explain one student: vì sao mô hình dự đoán như vậy?

Chọn **một sinh viên trong test set** và dùng `decision_path()` để xem mẫu đó đi qua những node nào.

### Nhiệm vụ

Viết lại đường đi theo dạng:

```text
Node 0: Attendance <= ...
→ đi trái/phải
Node ...
→ ...
Leaf: Needs_Support = ...
```

Sau đó giải thích bằng ngôn ngữ tự nhiên.

**[TRẢ LỜI]**
- Prediction = ...
- Actual = ...
- Đường đi quan trọng nhất = ...
- Tôi có thấy prediction này hợp lý theo logic của cây không? Vì sao?

In [ ]:
student = X_test.iloc[[0]]
actual = int(y_test.iloc[0])
prediction = int(best_tree.predict(student)[0])

display(student)
print("Actual:", actual)
print("Prediction:", prediction)

node_indicator = best_tree.decision_path(student)
leaf_id = best_tree.apply(student)[0]

node_index = node_indicator.indices[
    node_indicator.indptr[0]:node_indicator.indptr[1]
]

print("\nDecision path:")
for node_id in node_index:
    if node_id == leaf_id:
        values = best_tree.tree_.value[node_id][0]
        predicted_class = int(best_tree.classes_[np.argmax(values)])
        print(f"Leaf {node_id}: predicted class = {predicted_class}")
        continue

    feature_index = best_tree.tree_.feature[node_id]
    threshold = best_tree.tree_.threshold[node_id]
    feature_name = feature_cols[feature_index]
    feature_value = float(student.iloc[0, feature_index])

    direction = "<=" if feature_value <= threshold else ">"
    print(
        f"Node {node_id}: {feature_name}={feature_value:.2f} "
        f"{direction} {threshold:.2f}"
    )

# Mission 9 — Model Showdown: K-NN vs Naive Bayes vs Decision Tree

So sánh ba cách suy luận trên **cùng test set**:

| Model | Cách ra quyết định |
|---|---|
| K-NN | khoảng cách → hàng xóm → bỏ phiếu |
| GaussianNB | prior + likelihood → posterior |
| Decision Tree | câu hỏi → nhánh → leaf |

K-NN dùng `StandardScaler` như Lab 02.  
GaussianNB và Decision Tree dùng dữ liệu gốc.

**[TRẢ LỜI]**
1. Nếu ưu tiên không bỏ sót sinh viên cần hỗ trợ, metric nào quan trọng?
2. Model nào có Recall cao hơn trên test set này?
3. Model nào dễ giải thích trực quan nhất?
4. Một model Accuracy cao hơn một chút nhưng khó giải thích hơn có chắc phù hợp hơn cho Learning Analytics không? Giải thích.

In [ ]:
# K-NN reference — same pipeline as Lab 02
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn = KNeighborsClassifier(n_neighbors=5, metric="euclidean")
knn.fit(X_train_scaled, y_train)
knn_result, knn_pred = evaluate_binary(
    "K-NN k=5 scaled",
    knn,
    X_test_scaled,
    y_test,
    show_cm=False
)

# Gaussian Naive Bayes — same pipeline as Lab 03
gnb = GaussianNB()
gnb.fit(X_train, y_train)
nb_result, nb_pred = evaluate_binary(
    "GaussianNB",
    gnb,
    X_test,
    y_test,
    show_cm=False
)

# Best Decision Tree
dt_result, dt_pred = evaluate_binary(
    "Decision Tree",
    best_tree,
    X_test,
    y_test,
    show_cm=False
)

showdown = pd.DataFrame([knn_result, nb_result, dt_result])
display(showdown[
    ["model", "accuracy", "precision", "recall", "f1", "fn", "fp"]
])

# Optional Challenge — Decision Tree Regression

Decision Tree không chỉ dùng cho classification.

Nếu muốn thử thêm, dùng California Housing với `DecisionTreeRegressor`.

> Mặc định `RUN_OPTIONAL=False` để GitHub Actions không phụ thuộc vào việc tải thêm dataset.

**[TRẢ LỜI]**
- Khi target chuyển từ nhãn lớp sang giá trị liên tục, leaf node biểu diễn điều gì?
- Metric đánh giá regression có còn là Accuracy/Recall không?

In [ ]:
RUN_OPTIONAL = False

if RUN_OPTIONAL:
    from sklearn.datasets import fetch_california_housing
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    housing = fetch_california_housing(as_frame=True)
    X_h = housing.data
    y_h = housing.target

    Xh_train, Xh_test, yh_train, yh_test = train_test_split(
        X_h, y_h, test_size=0.25, random_state=42
    )

    reg = DecisionTreeRegressor(max_depth=5, random_state=42)
    reg.fit(Xh_train, yh_train)
    pred_h = reg.predict(Xh_test)

    print("MAE:", mean_absolute_error(yh_test, pred_h))
    print("RMSE:", mean_squared_error(yh_test, pred_h) ** 0.5)
    print("R2:", r2_score(yh_test, pred_h))
else:
    print("Optional Challenge skipped. Set RUN_OPTIONAL=True to run it.")

# Final — Model Reflection Card

## 1. Decision Tree ra quyết định như thế nào?
**[VIẾT]**

## 2. Root Node có ý nghĩa gì?
**[VIẾT]**

## 3. Gini và Entropy được dùng để làm gì?
**[VIẾT]**

## 4. Vì sao cây quá sâu có thể gây overfitting?
**[VIẾT]**

## 5. Trong case study `Needs_Support`, False Negative có ý nghĩa gì?
**[VIẾT]**

## 6. Decision Tree cho biết điều gì rõ hơn K-NN và Naive Bayes?
**[VIẾT]**

## 7. Performance và interpretability có thể đánh đổi với nhau như thế nào?
**[VIẾT]**

## 8. Một câu chốt
> Love AI giúp tôi hiểu cơ chế của Decision Tree; Student Performance Factors giúp tôi thấy rằng...

**[VIẾT]**

## Trước khi push

```bash
python scripts/download_data.py
python tests/check_lab04.py
git add .
git commit -m "Complete Lab 04 Decision Tree"
git push
```

Sau khi push, kiểm tra tab **Actions** và đảm bảo workflow `Lab 04 Check` chạy thành công.